# Day G4 — 분자 표현과 활성 예측: SMILES, 지문, QSAR, 분자 생성

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/G-ai-biomed/day-G4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/G-ai-biomed/day-G4.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q rdkit

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 실제 약물의 SMILES, 기술자, 지문 유사도

In [ ]:
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, QED, rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")

drugs = {   # 이름: SMILES (입체 정보는 일부 생략)
    "levodopa": "N[C@@H](Cc1ccc(O)c(O)c1)C(=O)O",
    "dopamine": "NCCc1ccc(O)c(O)c1",
    "carbidopa": "CC(Cc1ccc(O)c(O)c1)(NN)C(=O)O",
    "selegiline": "C#CCN(C)C(C)Cc1ccccc1",
    "rasagiline": "C#CCN[C@@H]1CCc2ccccc21",
    "donepezil": "COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2",
    "rivastigmine": "CCN(C)C(=O)Oc1cccc(c1)[C@H](C)N(C)C",
    "memantine": "CC12CC3CC(C)(C1)CC(N)(C3)C2",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
}
mols = {k: Chem.MolFromSmiles(s) for k, s in drugs.items()}
print(f"{'이름':<13}{'분자식':<12}{'MW':>7}{'logP':>6}{'HBD':>4}{'HBA':>4}{'TPSA':>7}{'QED':>6}")
for k, m in mols.items():
    print(f"{k:<13}{rdMolDescriptors.CalcMolFormula(m):<12}{Descriptors.MolWt(m):7.1f}"
          f"{Crippen.MolLogP(m):6.2f}{rdMolDescriptors.CalcNumHBD(m):4d}{rdMolDescriptors.CalcNumHBA(m):4d}"
          f"{rdMolDescriptors.CalcTPSA(m):7.1f}{QED.qed(m):6.2f}")

# 같은 분자, 다른 SMILES → canonical SMILES는 하나
for s in ["OC(=O)C(N)Cc1ccc(O)c(O)c1", "c1cc(O)c(O)cc1CC(N)C(O)=O"]:
    print(s, "→", Chem.MolToSmiles(Chem.MolFromSmiles(s)))
print("문법이 틀린 SMILES:", Chem.MolFromSmiles("c1ccc(cc1"))

# Morgan(ECFP4 계열) 지문과 Tanimoto 유사도
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = {k: gen.GetFingerprint(m) for k, m in mols.items()}
print("levodopa 기준 Tanimoto 유사도:")
for k in drugs:
    print(f"  {k:<13}{DataStructs.TanimotoSimilarity(fps['levodopa'], fps[k]):.2f}")

### 예시 2 — 가상 QSAR: 무작위 분할 vs 골격 분할, 그리고 적용 영역

In [ ]:
import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Crippen, rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error
RDLogger.DisableLog("rdApp.*")
rng = np.random.default_rng(0)

# (1) 단위 변환: IC50(nM) → pIC50 = -log10(IC50 [M]). 클수록 강한 활성.
for ic50_nM in [1, 10, 250, 10000]:
    print(f"IC50 {ic50_nM:>6} nM → pIC50 {-np.log10(ic50_nM * 1e-9):.2f}")

# (2) 가상 화합물 시리즈: 골격(scaffold) 6개 × 치환기 2자리. 활성은 '가상 규칙 + 잡음'.
scaffolds = {   # {a}, {b}에 치환기가 들어간다. 오른쪽 숫자는 골격 고유의 가상 활성 보정값
    "c1cc({a})ccc1C(=O)Nc1ccc({b})cc1": 0.0,
    "c1cc({a})ccc1S(=O)(=O)Nc1ccc({b})cc1": -0.8,
    "c1cc({a})ccc1CNC(=O)c1ccc({b})nc1": 0.6,
    "c1cc({a})ccc1Oc1ncc({b})cn1": 1.0,
    "c1cc({a})ccc1C1CCN(C1)c1ccc({b})cc1": -0.4,
    "c1cc({a})ccc1-c1nc2cc({b})ccc2o1": 1.4,
}
subs = ["[H]", "F", "Cl", "Br", "C", "CC", "OC", "C(F)(F)F", "C#N", "O", "N"]
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
smiles, y, fps = [], [], []
for tpl, offset in scaffolds.items():
    for a in subs:
        for b in subs:
            m = Chem.RemoveHs(Chem.MolFromSmiles(tpl.format(a=a, b=b)))
            logp = Crippen.MolLogP(m)
            hal = sum(at.GetSymbol() in ("F", "Cl", "Br") for at in m.GetAtoms())
            act = 6.0 + offset + 0.35 * logp - 0.15 * (logp - 3.5) ** 2 + 0.2 * hal + rng.normal(0, 0.3)
            smiles.append(Chem.MolToSmiles(m)); y.append(act)
            fps.append(gen.GetFingerprint(m))
X = np.array([list(fp) for fp in fps], dtype=np.uint8)
y = np.array(y)

def murcko(smi):
    """단순화한 Bemis–Murcko 골격: 고리와 고리를 잇는 사슬만 남기고, 끝에 매달린 원자를 반복해 지운다.
    (보통은 rdkit.Chem.Scaffolds.MurckoScaffold를 쓰지만, 이 환경의 RDKit 판에서 오류가 나 직접 구현)"""
    rw = Chem.RWMol(Chem.MolFromSmiles(smi))
    while True:
        ends = [a.GetIdx() for a in rw.GetAtoms() if a.GetDegree() <= 1 and not a.IsInRing()]
        if not ends:
            break
        for i in sorted(ends, reverse=True):
            rw.RemoveAtom(i)
    m = rw.GetMol()
    for a in m.GetAtoms():          # 지운 자리 때문에 남은 수소 수 정보를 다시 계산하게 한다
        a.SetNoImplicit(False); a.SetNumExplicitHs(0)
    Chem.SanitizeMol(m)
    return Chem.MolToSmiles(m)

assert murcko("CCOc1ccc(Cl)cc1C(=O)NCc1ccccn1") == murcko("c1ccccc1C(=O)NCc1ncccc1")

# 실제 작업처럼 SMILES에서 Murcko 골격을 계산해 묶음(group) 이름으로 쓴다
scaf = [murcko(s) for s in smiles]
groups = np.unique(scaf, return_inverse=True)[1]
print("화합물", len(y), "개 | 서로 다른 Murcko 골격", groups.max() + 1, "개")

rf = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)
for name, cv, g in [("무작위 5-fold", KFold(5, shuffle=True, random_state=0), None),
                    ("골격 단위 GroupKFold", GroupKFold(n_splits=6), groups)]:
    pred = cross_val_predict(rf, X, y, cv=cv, groups=g)
    base = cross_val_predict(DummyRegressor(), X, y, cv=cv, groups=g)
    print(f"{name:<20} RF: R² {r2_score(y, pred):.2f}, RMSE {np.sqrt(mean_squared_error(y, pred)):.2f}"
          f" | 평균만 답하는 기준선 RMSE {np.sqrt(mean_squared_error(y, base)):.2f}")
    if g is not None:
        scaf_pred = pred

# (3) 적용 영역: 학습 화합물과 얼마나 닮았나(최대 Tanimoto) vs 예측 오차
err = np.abs(scaf_pred - y)
maxsim = np.array([max(DataStructs.BulkTanimotoSimilarity(fps[i], [fps[j] for j in np.where(groups != groups[i])[0]]))
                   for i in range(len(y))])
cuts = np.quantile(maxsim, [0, 1/3, 2/3, 1])          # 화합물을 유사도 순으로 셋으로 나눈다
for i, name in enumerate(["낮음", "중간", "높음"]):
    sel = (maxsim >= cuts[i]) & ((maxsim < cuts[i + 1]) | (i == 2))
    print(f"학습셋과의 유사도 {name} ({cuts[i]:.2f}–{cuts[i+1]:.2f}): 화합물 {sel.sum():>3}개, 평균 절대오차 {err[sel].mean():.2f}")

### 예시 3 — 엉성한 생성기와 validity·uniqueness·novelty

In [ ]:
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import QED
RDLogger.DisableLog("rdApp.*")
rng = np.random.default_rng(0)

# 아주 엉성한 '생성기': 학습 SMILES의 글자 하나를 무작위로 바꾸거나 지운다
train = ["NCCc1ccc(O)c(O)c1", "C#CCN(C)C(C)Cc1ccccc1", "CCN(C)C(=O)Oc1cccc(c1)C(C)N(C)C",
         "COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2", "CC12CC3CC(C)(C1)CC(N)(C3)C2"]
alphabet = list("CNOcn()=1#")
train_canon = {Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in train}

def mutate(s):
    i = rng.integers(len(s))
    return s[:i] + (rng.choice(alphabet) if rng.random() < 0.7 else "") + s[i + 1:]

samples = [mutate(train[rng.integers(len(train))]) for _ in range(500)]
mols = [Chem.MolFromSmiles(s) for s in samples]
valid = [Chem.MolToSmiles(m) for m in mols if m is not None]
unique = set(valid)
novel = unique - train_canon
print(f"생성 500개 → 유효(validity) {len(valid) / 500:.0%}, "
      f"유효 중 고유(uniqueness) {len(unique) / len(valid):.0%}, "
      f"고유 중 새로움(novelty) {len(novel) / len(unique):.0%}")
q = sorted(((QED.qed(Chem.MolFromSmiles(s)), s) for s in novel), reverse=True)[:3]
print("QED가 높은 새 분자 3개:")
for score, s in q:
    print(f"  {score:.2f}  {s}")